# Chroma CRUD Operations


This notebook walks through create, read, update, and delete operations with a local Chroma vector store

In [1]:
import os
import shutil
from pathlib import Path
from uuid import uuid4

from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings

In [2]:
# Resolve the project root so the notebook works from either the repo root or the notebooks folder.
project_root = Path.cwd()
if project_root.name == "notebooks":
    project_root = project_root.parent

project_root

WindowsPath('d:/GenAI/Jayeeta/Advanced-RAG/5.Vector')

In [3]:
# Load environment variables from the local .env file.
dotenv_path = project_root.parent / ".env"
load_dotenv(dotenv_path=dotenv_path)

if not os.getenv("OPENAI_API_KEY"):
    raise ValueError("Please add your OPENAI_API_KEY to the .env file before running this notebook.")

print(f"Loaded environment from: {dotenv_path}")

Loaded environment from: d:\GenAI\Jayeeta\Advanced-RAG\.env


In [4]:
# Use a fixed collection name and persistence path so each rerun is predictable.
collection_name = "demo_2"
persist_directory = project_root / "notebooks" / "chroma_langchain_db"

print(f"Collection name: {collection_name}")
print(f"Persist directory: {persist_directory}")

Collection name: demo_2
Persist directory: d:\GenAI\Jayeeta\Advanced-RAG\5.Vector\notebooks\chroma_langchain_db


In [5]:
# Start fresh so the CRUD flow produces the same result each time.
if persist_directory.exists():
    shutil.rmtree(persist_directory)
    print("Removed the old Chroma directory.")
else:
    print("No previous Chroma directory was found.")

No previous Chroma directory was found.


In [13]:
# Create the embedding model and connect it to a persistent Chroma store.
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")

vector_store = Chroma(
    collection_name=collection_name,
    embedding_function=embeddings,
    persist_directory=str(persist_directory),
)

print("Vector store is ready.")

Vector store is ready.


# 2.Add samll Helper Functions

In [6]:
def preview_text(text, limit=80):
    """Return a short preview for cleaner notebook output."""
    if len(text) <= limit:
        return text
    return text[:limit] + "..."


def print_documents(title, docs):
    """Print Document objects in a beginner-friendly format."""
    print(title)
    for index, doc in enumerate(docs, start=1):
        print(f"{index}. id={doc.id}")
        print(f"   topic={doc.metadata.get('topic')} | doc_number={doc.metadata.get('doc_number')}")
        print(f"   content={doc.page_content}")
    print()

In [7]:
# Keep the raw sample data separate from the Document objects so it is easier to read.
document_examples = [
    {
        "topic": "AI",
        "doc_number": 1,
        "text": "Artificial intelligence helps machines perform tasks that usually need human reasoning.",
    },
    {
        "topic": "AI",
        "doc_number": 2,
        "text": "AI systems can analyze patterns in data to support predictions and automation.",
    },
    {
        "topic": "AI",
        "doc_number": 3,
        "text": "Responsible AI development includes fairness, transparency, and safety checks.",
    },
    {
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG combines retrieval with generation so the model can answer using external knowledge.",
    },
    {
        "topic": "RAG",
        "doc_number": 5,
        "text": "A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.",
    },
    {
        "topic": "RAG",
        "doc_number": 6,
        "text": "Vector stores are important in RAG because they make semantic search over embedded documents possible.",
    },
    {
        "topic": "LLM",
        "doc_number": 7,
        "text": "LLMs generate text by predicting likely next tokens from patterns learned during training.",
    },
    {
        "topic": "LLM",
        "doc_number": 8,
        "text": "Prompt design can improve how clearly an LLM follows instructions and returns useful answers.",
    },
    {
        "topic": "Cricket",
        "doc_number": 9,
        "text": "Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.",
    },
    {
        "topic": "Cricket",
        "doc_number": 10,
        "text": "A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.",
    },
]

print(f"Prepared {len(document_examples)} document examples.")

Prepared 10 document examples.


In [8]:
for doc in document_examples:
    print(doc)
    print()

{'topic': 'AI', 'doc_number': 1, 'text': 'Artificial intelligence helps machines perform tasks that usually need human reasoning.'}

{'topic': 'AI', 'doc_number': 2, 'text': 'AI systems can analyze patterns in data to support predictions and automation.'}

{'topic': 'AI', 'doc_number': 3, 'text': 'Responsible AI development includes fairness, transparency, and safety checks.'}

{'topic': 'RAG', 'doc_number': 4, 'text': 'RAG combines retrieval with generation so the model can answer using external knowledge.'}

{'topic': 'RAG', 'doc_number': 5, 'text': 'A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'}

{'topic': 'RAG', 'doc_number': 6, 'text': 'Vector stores are important in RAG because they make semantic search over embedded documents possible.'}

{'topic': 'LLM', 'doc_number': 7, 'text': 'LLMs generate text by predicting likely next tokens from patterns learned during training.'}

{'topic': 'LLM', 'doc_number': 8, 'text': 'Prompt des

In [9]:
print(uuid4())

56023d80-e55a-4eaf-b6fa-66643baea437


In [10]:
# Convert the sample data into LangChain Document objects.
documents = [
    Document(
        id=str(uuid4()),
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in document_examples
]

print_documents("Dummy documents prepared:", documents)

Dummy documents prepared:
1. id=5b40bbc2-dc74-4743-bc35-22b013ee0288
   topic=AI | doc_number=1
   content=Artificial intelligence helps machines perform tasks that usually need human reasoning.
2. id=e05b7ef7-cc12-433d-8a71-2a9ca779ea24
   topic=AI | doc_number=2
   content=AI systems can analyze patterns in data to support predictions and automation.
3. id=bbb20cec-e737-43b8-a422-8ac74a8c5eb0
   topic=AI | doc_number=3
   content=Responsible AI development includes fairness, transparency, and safety checks.
4. id=5f474307-7f84-4987-9445-7c08d2b292db
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
5. id=95529c4e-f71a-4d73-8614-3b409785bef0
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.
6. id=e0c53181-c6fa-459c-a1a2-662699822b72
   topic=RAG | doc_number=6
   content=Vector stores are important in RAG because they mak

In [11]:

documents[0].id

'5b40bbc2-dc74-4743-bc35-22b013ee0288'

In [14]:
# Insert the documents into Chroma. Chroma creates embeddings during this step.
document_ids = vector_store.add_documents(documents)

print("Inserted document ids:")
for doc_id in document_ids:
    print(doc_id)

print(f"\nTotal inserted documents: {len(document_ids)}")

Inserted document ids:
5b40bbc2-dc74-4743-bc35-22b013ee0288
e05b7ef7-cc12-433d-8a71-2a9ca779ea24
bbb20cec-e737-43b8-a422-8ac74a8c5eb0
5f474307-7f84-4987-9445-7c08d2b292db
95529c4e-f71a-4d73-8614-3b409785bef0
e0c53181-c6fa-459c-a1a2-662699822b72
44f6345e-891a-4686-8028-b71fb0f9006a
0132b88b-8764-4274-8e56-3de7997d2031
c7a127b6-51de-450b-b85d-891871a24bbb
e868845d-2f01-4acc-9d55-7ddde7d5258a

Total inserted documents: 10


## 4. Read the Stored Data Back

In [20]:
# The get() method returns the low-level Chroma record structure.
raw_records = vector_store.get(include=["embeddings", "metadatas", "documents"])
raw_records.keys()

dict_keys(['ids', 'embeddings', 'documents', 'uris', 'included', 'data', 'metadatas'])

In [21]:
raw_records

{'ids': ['5b40bbc2-dc74-4743-bc35-22b013ee0288',
  'e05b7ef7-cc12-433d-8a71-2a9ca779ea24',
  'bbb20cec-e737-43b8-a422-8ac74a8c5eb0',
  '5f474307-7f84-4987-9445-7c08d2b292db',
  '95529c4e-f71a-4d73-8614-3b409785bef0',
  'e0c53181-c6fa-459c-a1a2-662699822b72',
  '44f6345e-891a-4686-8028-b71fb0f9006a',
  '0132b88b-8764-4274-8e56-3de7997d2031',
  'c7a127b6-51de-450b-b85d-891871a24bbb',
  'e868845d-2f01-4acc-9d55-7ddde7d5258a'],
 'embeddings': array([[ 0.00489426,  0.02098083,  0.0165863 , ...,  0.00072622,
         -0.0164032 ,  0.02793884],
        [-0.01451111, -0.00540543,  0.03039551, ..., -0.02856445,
         -0.00189972,  0.04272461],
        [ 0.02268982,  0.01530457,  0.04550171, ...,  0.02297974,
          0.00804138, -0.01586914],
        ...,
        [ 0.00801849,  0.02313232,  0.02172852, ..., -0.02067566,
         -0.01786804,  0.01333618],
        [ 0.00880432,  0.06268311,  0.09967041, ..., -0.01580811,
         -0.01374817,  0.0368042 ],
        [ 0.0178833 ,  0.08526611, 

In [27]:
print(raw_records["embeddings"][0:2, 0:20])

[[ 0.00489426  0.02098083  0.0165863   0.0045929   0.0324707  -0.00642014
  -0.02862549  0.06011963  0.00505447  0.04025269  0.02565002 -0.01589966
  -0.00541306 -0.02372742  0.00824738 -0.02932739 -0.02622986 -0.03842163
   0.03295898  0.00601578]
 [-0.01451111 -0.00540543  0.03039551  0.01064301  0.04437256  0.00770187
  -0.01461029  0.02972412 -0.0057869   0.05966187  0.00088644 -0.05419922
  -0.00701904 -0.04064941 -0.00610733  0.01528931 -0.00791931  0.04486084
  -0.02912903 -0.01476288]]


In [23]:
print(f"Total records in collection: {len(raw_records['ids'])}")
print("First three ids from get():")
for doc_id in raw_records["ids"][:3]:
    print(doc_id)

Total records in collection: 10
First three ids from get():
5b40bbc2-dc74-4743-bc35-22b013ee0288
e05b7ef7-cc12-433d-8a71-2a9ca779ea24
bbb20cec-e737-43b8-a422-8ac74a8c5eb0


In [24]:
# Pick a few ids so we can read them back in a higher-level format.
selected_ids = document_ids[-3:]
selected_ids

['0132b88b-8764-4274-8e56-3de7997d2031',
 'c7a127b6-51de-450b-b85d-891871a24bbb',
 'e868845d-2f01-4acc-9d55-7ddde7d5258a']

In [25]:
# get_by_ids() returns LangChain Document objects instead of the raw Chroma dictionary.
selected_documents = vector_store.get_by_ids(selected_ids)
print_documents("Documents fetched with get_by_ids():", selected_documents)

Documents fetched with get_by_ids():
1. id=0132b88b-8764-4274-8e56-3de7997d2031
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
2. id=c7a127b6-51de-450b-b85d-891871a24bbb
   topic=Cricket | doc_number=9
   content=Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.
3. id=e868845d-2f01-4acc-9d55-7ddde7d5258a
   topic=Cricket | doc_number=10
   content=A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.



In [26]:
print(selected_documents)

[Document(id='0132b88b-8764-4274-8e56-3de7997d2031', metadata={'topic': 'LLM', 'doc_number': 8}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'), Document(id='c7a127b6-51de-450b-b85d-891871a24bbb', metadata={'doc_number': 9, 'topic': 'Cricket'}, page_content='Cricket teams score runs through batting partnerships, boundaries, and quick running between the wickets.'), Document(id='e868845d-2f01-4acc-9d55-7ddde7d5258a', metadata={'topic': 'Cricket', 'doc_number': 10}, page_content='A cricket bowler can pressure batters with pace, swing, spin, and accurate line and length.')]


## 5. Run a Similarity Search

In [29]:
query = "How does RAG help an LLM answer questions using outside knowledge?"
query

'How does RAG help an LLM answer questions using outside knowledge?'

In [30]:
search_results = vector_store.similarity_search(query, k=3)
print(f"Query: {query}\n")
print_documents("Similarity search results:", search_results)

Query: How does RAG help an LLM answer questions using outside knowledge?

Similarity search results:
1. id=5f474307-7f84-4987-9445-7c08d2b292db
   topic=RAG | doc_number=4
   content=RAG combines retrieval with generation so the model can answer using external knowledge.
2. id=0132b88b-8764-4274-8e56-3de7997d2031
   topic=LLM | doc_number=8
   content=Prompt design can improve how clearly an LLM follows instructions and returns useful answers.
3. id=95529c4e-f71a-4d73-8614-3b409785bef0
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



In [31]:
search_results

[Document(id='5f474307-7f84-4987-9445-7c08d2b292db', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
 Document(id='0132b88b-8764-4274-8e56-3de7997d2031', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
 Document(id='95529c4e-f71a-4d73-8614-3b409785bef0', metadata={'topic': 'RAG', 'doc_number': 5}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.')]

In [32]:
vector_store.similarity_search_with_score(query=query, k=4)

[(Document(id='5f474307-7f84-4987-9445-7c08d2b292db', metadata={'topic': 'RAG', 'doc_number': 4}, page_content='RAG combines retrieval with generation so the model can answer using external knowledge.'),
  0.8066883087158203),
 (Document(id='0132b88b-8764-4274-8e56-3de7997d2031', metadata={'doc_number': 8, 'topic': 'LLM'}, page_content='Prompt design can improve how clearly an LLM follows instructions and returns useful answers.'),
  0.9389731884002686),
 (Document(id='95529c4e-f71a-4d73-8614-3b409785bef0', metadata={'doc_number': 5, 'topic': 'RAG'}, page_content='A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.'),
  1.075084924697876),
 (Document(id='44f6345e-891a-4686-8028-b71fb0f9006a', metadata={'doc_number': 7, 'topic': 'LLM'}, page_content='LLMs generate text by predicting likely next tokens from patterns learned during training.'),
  1.1305420398712158)]

## 6. Update Existing Documents

In [33]:
# We will update one RAG document and one LLM document.
ids_to_update = [document_ids[3], document_ids[7]]
ids_to_update

['5f474307-7f84-4987-9445-7c08d2b292db',
 '0132b88b-8764-4274-8e56-3de7997d2031']

In [34]:
# Keep the replacement text separate so the update step stays easy to follow.
updated_examples = [
    {
        "id": ids_to_update[0],
        "topic": "RAG",
        "doc_number": 4,
        "text": "RAG improves answer quality by retrieving relevant context before the language model generates a response.",
    },
    {
        "id": ids_to_update[1],
        "topic": "LLM",
        "doc_number": 8,
        "text": "Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.",
    },
]

updated_documents = [
    Document(
        id=item["id"],
        page_content=item["text"],
        metadata={"topic": item["topic"], "doc_number": item["doc_number"]},
    )
    for item in updated_examples
]

print_documents("Updated document content:", updated_documents)

Updated document content:
1. id=5f474307-7f84-4987-9445-7c08d2b292db
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=0132b88b-8764-4274-8e56-3de7997d2031
   topic=LLM | doc_number=8
   content=Well-written prompts help an LLM stay focused, follow instructions, and produce more reliable outputs.



In [35]:
print([doc.page_content for doc in documents if doc.id in ids_to_update])

['RAG combines retrieval with generation so the model can answer using external knowledge.', 'Prompt design can improve how clearly an LLM follows instructions and returns useful answers.']


In [36]:
vector_store.update_documents(ids=ids_to_update, documents=updated_documents)

print("Updated these ids:")
for doc_id in ids_to_update:
    print(doc_id)

Updated these ids:
5f474307-7f84-4987-9445-7c08d2b292db
0132b88b-8764-4274-8e56-3de7997d2031


In [40]:
# Read the updated records back from Chroma to confirm the new values were stored.
updated_raw_records = vector_store.get(ids=ids_to_update)

print("Raw records returned by get(ids=ids_to_update):")
for doc_id, document_text, metadata in zip(
    updated_raw_records["ids"],
    updated_raw_records["documents"],
    updated_raw_records["metadatas"],
):
    print(f"id={doc_id}")
    print(f"metadata={metadata}")
    print(f"content={preview_text(document_text)}")
    print()

Raw records returned by get(ids=ids_to_update):
id=5f474307-7f84-4987-9445-7c08d2b292db
metadata={'topic': 'RAG', 'doc_number': 4}
content=RAG improves answer quality by retrieving relevant context before the language m...

id=0132b88b-8764-4274-8e56-3de7997d2031
metadata={'doc_number': 8, 'topic': 'LLM'}
content=Well-written prompts help an LLM stay focused, follow instructions, and produce ...



In [38]:
updated_query = "How can retrieved context improve an LLM response in RAG?"
updated_query

'How can retrieved context improve an LLM response in RAG?'

In [39]:
updated_search_results = vector_store.similarity_search(updated_query, k=2)
print(f"Updated query: {updated_query}\n")
print_documents("Similarity search after update:", updated_search_results)

Updated query: How can retrieved context improve an LLM response in RAG?

Similarity search after update:
1. id=5f474307-7f84-4987-9445-7c08d2b292db
   topic=RAG | doc_number=4
   content=RAG improves answer quality by retrieving relevant context before the language model generates a response.
2. id=95529c4e-f71a-4d73-8614-3b409785bef0
   topic=RAG | doc_number=5
   content=A retriever in a RAG pipeline finds relevant chunks before the language model generates an answer.



## 7. Delete Documents

In [41]:
# Delete the two cricket examples so the final collection is smaller.
ids_to_delete = [document_ids[8], document_ids[9]]
ids_to_delete

['c7a127b6-51de-450b-b85d-891871a24bbb',
 'e868845d-2f01-4acc-9d55-7ddde7d5258a']

In [42]:
vector_store.delete(ids=ids_to_delete)

print("Deleted these ids:")
for doc_id in ids_to_delete:
    print(doc_id)

Deleted these ids:
c7a127b6-51de-450b-b85d-891871a24bbb
e868845d-2f01-4acc-9d55-7ddde7d5258a


In [43]:
remaining_records = vector_store.get()
remaining_ids = remaining_records["ids"]

print(f"Remaining document count: {len(remaining_ids)}")
print("Remaining ids:")
for doc_id in remaining_ids:
    print(doc_id)

print("\nDeleted ids still present?")
for doc_id in ids_to_delete:
    print(f"{doc_id}: {doc_id in remaining_ids}")

Remaining document count: 8
Remaining ids:
5b40bbc2-dc74-4743-bc35-22b013ee0288
e05b7ef7-cc12-433d-8a71-2a9ca779ea24
bbb20cec-e737-43b8-a422-8ac74a8c5eb0
5f474307-7f84-4987-9445-7c08d2b292db
95529c4e-f71a-4d73-8614-3b409785bef0
e0c53181-c6fa-459c-a1a2-662699822b72
44f6345e-891a-4686-8028-b71fb0f9006a
0132b88b-8764-4274-8e56-3de7997d2031

Deleted ids still present?
c7a127b6-51de-450b-b85d-891871a24bbb: False
e868845d-2f01-4acc-9d55-7ddde7d5258a: False


In [44]:
print([doc.metadata["topic"] for doc in documents if doc.id in remaining_ids])

['AI', 'AI', 'AI', 'RAG', 'RAG', 'RAG', 'LLM', 'LLM']
